# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

We load the Java API of `Z3` from *Maven Central* and import the package `com.microsoft.z3`.  All `Z3` objects are created by a *context*, which we call `Z3`.

In [ ]:
%maven tools.aqua:z3-turnkey:4.14.1
import com.microsoft.z3.*;

In [ ]:
Context Z3 = new Context();

## Auxiliary Functions

The symbolic transition system uses 3 variables.
* `M` is the number of missionaries on the left side of the river,
* `C` is the number of infidels on the left side of the river,
* `B` is the number of boats on the left side of the river.

We define these variables using `Z3.mkIntConst`.

In [ ]:
IntExpr M = Z3.mkIntConst("M");  IntExpr MX = Z3.mkIntConst("MX");
IntExpr C = Z3.mkIntConst("C");  IntExpr CX = Z3.mkIntConst("CX");
IntExpr B = Z3.mkIntConst("B");  IntExpr BX = Z3.mkIntConst("BX");

The function `start` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a `Z3` formula that specifies that everybody is on the western shore.
$$ \texttt{M} = 3 \wedge \texttt{C} = 3 \wedge \texttt{B} = 1 $$

In [ ]:
BoolExpr start(IntExpr M, IntExpr C, IntExpr B) {
    return Z3.mkAnd(Z3.mkEq(M, Z3.mkInt(3)), Z3.mkEq(C, Z3.mkInt(3)), Z3.mkEq(B, Z3.mkInt(1)));
}

`Z3` expressions are printed in the syntax of the language *SMT-LIB*, which uses prefix notation.

In [ ]:
start(M, C, B).toString()

The function `goal` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a formula that specifies that everybody is on the eastern shore.
$$ \texttt{M} = 0 \wedge \texttt{C} = 0 \wedge \texttt{B} = 0 $$

In [ ]:
BoolExpr goal(IntExpr M, IntExpr C, IntExpr B) {
    return Z3.mkAnd(Z3.mkEq(M, Z3.mkInt(0)), Z3.mkEq(C, Z3.mkInt(0)), Z3.mkEq(B, Z3.mkInt(0)));
}

The function `invariant` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a list of formulas. The conjunction of these formulas is `true` if there is no problem on either shore of the river.
There is no problem if either of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e. 
  $\texttt{M} = 0$.  
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M} = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of 
  infidels on that side, i.e. $\texttt{M} = \texttt{C}$.  Then the numbers of 
  missionaries and infidels have to match on the eastern shore as well.
* Furthermore, the number of missionaries and Cannibals is an element of the set $\{0,1,2,3\}$:
  $$ 0 \leq \texttt{M} \leq 3 \;\wedge\; 0 \leq \texttt{C} \leq 3$$

In [ ]:
List<BoolExpr> invariant(IntExpr M, IntExpr C, IntExpr B) {
    return List.of(Z3.mkOr(Z3.mkEq(M, Z3.mkInt(0)), Z3.mkEq(M, Z3.mkInt(3)), Z3.mkEq(M, C)),  // M = 0 ∨ M = 3 ∨ M = C
                   Z3.mkGe(M, Z3.mkInt(0)), Z3.mkLe(M, Z3.mkInt(3)),                          // 0 ≤ M ≤ 3
                   Z3.mkGe(C, Z3.mkInt(0)), Z3.mkLe(C, Z3.mkInt(3)),                          // 0 ≤ C ≤ 3
                   Z3.mkGe(B, Z3.mkInt(0)), Z3.mkLe(B, Z3.mkInt(1)));                         // 0 ≤ B ≤ 1
}

In [ ]:
for (var constraint : invariant(M, C, B)) {
    System.out.println(constraint);
}

The function `boatOK` takes 6 arguments:
* `Ma` is the number of missionaries on the western shore before the crossing.
* `Ca` is the number of infidels on the western shore before the crossing.
* `Ba` is the number of boats on the western shore before the crossing. 
* `Mb` is the number of missionaries on the western shore after the crossing.
* `Cb` is the number of infidels on the western shore after the crossing.
* `Bb` is the number of boats on the western shore after the crossing.

The function returns a formula that checks that if the boat 
crosses the river from the western shore to the eastern shore, then the number of people on the boat is between $1$ and $2$
and the number of missionaries and cannibals on the western shore can't increase:
$$ \mathtt{Ba} = 1 \;\rightarrow\; 1 \leq \mathtt{Ma} - \mathtt{Mb} + \mathtt{Ca} - \mathtt{Cb} \leq 2 \,\wedge\, \mathtt{Mb} \leq \mathtt{Ma} \,\wedge\, \mathtt{Cb} \leq \mathtt{Ca}$$
Here `Ma - Mb` is the number of missionaries on the boat and  `Ca - Cb` is the number of infidels on the boat.  The formula 
$1 \leq \mathtt{Ma} - \mathtt{Mb} + \mathtt{Ca} - \mathtt{Cb} \leq 2$ expresses that there is at least one but not more than two passengers on the boat.

In [ ]:
BoolExpr boatOK(IntExpr Ma, IntExpr Ca, IntExpr Ba, IntExpr Mb, IntExpr Cb, IntExpr Bb) {
    var numMis = Z3.mkSub(Ma, Mb);           // Ma - Mb
    var numCan = Z3.mkSub(Ca, Cb);           // Ca - Cb
    var all    = Z3.mkAdd(numMis, numCan);   // Ma - Mb + Ca - Cb
    // Ba = 1 → 1 ≤ Ma - Mb + Ca - Cb ≤ 2 ∧ Mb ≤ Ma ∧ Cb ≤ Ca
    return Z3.mkImplies(Z3.mkEq(Ba, Z3.mkInt(1)),
                        Z3.mkAnd(Z3.mkGe(all, Z3.mkInt(1)), Z3.mkLe(all, Z3.mkInt(2)),
                                 Z3.mkLe(Mb, Ma), Z3.mkLe(Cb, Ca)));
}

In [ ]:
boatOK(M, C, B, MX, CX, BX).toString()

The function `transition` takes 6 arguments:
* `Ma` is the number of missionaries on the western shore before the crossing.
* `Ca` is the number of infidels on the western shore before the crossing.
* `Ba` is the number of boats on the western shore before the crossing. 
* `Mb` is the number of missionaries on the western shore after the crossing.
* `Cb` is the number of infidels on the western shore after the crossing.
* `Bb` is the number of boats on the western shore after the crossing.

It returns a formula that expresses the following:
* The boat changes the sides in every transition:
  $$ \mathtt{Bb} = 1 - \mathtt{Ba} $$
* If the boat crosses the river from the western shore to the eastern shore,
  then there is at least one and at most two passengers and the numbers on the western shore do not increase.
* If the boat crosses the river from the eastern shore to the western shore,
  then there is at least one and at most two passengers and the numbers on the eastern shore do not increase.

In [ ]:
List<BoolExpr> transition(IntExpr Ma, IntExpr Ca, IntExpr Ba, IntExpr Mb, IntExpr Cb, IntExpr Bb) {
    return List.of(Z3.mkEq(Bb, Z3.mkSub(Z3.mkInt(1), Ba)),   // Bb = 1 - Ba
                   boatOK(Ma, Ca, Ba, Mb, Cb, Bb),
                   boatOK(Mb, Cb, Bb, Ma, Ca, Ba));
}

In [ ]:
transition(M, C, B, MX, CX, BX).toString()

The function `missionariesCSP` creates a CSP that tries to solve the problem with `n` crossings.  It creates a new `Solver` instance, adds all constraints and checks whether the constraints are satisfiable.  If they are, the function returns a map that maps the variables to their values.  Otherwise, it returns `null`.

The method `add` of a solver takes a variable number of arguments of type `BoolExpr`.  To add a list of constraints, we have to convert the list into an array via `toArray(BoolExpr[]::new)`.

In [ ]:
Map<String, Integer> missionariesCSP(int n) {
    Solver solver = Z3.mkSolver();
    List<IntExpr> Ms = new ArrayList<>(), Cs = new ArrayList<>(), Bs = new ArrayList<>();
    for (int i = 0; i <= n; ++i) {
        Ms.add(Z3.mkIntConst("M" + i));
        Cs.add(Z3.mkIntConst("C" + i));
        Bs.add(Z3.mkIntConst("B" + i));
    }
    solver.add(start(Ms.get(0), Cs.get(0), Bs.get(0)));
    solver.add(goal (Ms.get(n), Cs.get(n), Bs.get(n)));
    for (int i = 0; i < n; ++i) {
        solver.add(invariant (Ms.get(i), Cs.get(i), Bs.get(i)).toArray(BoolExpr[]::new));
        solver.add(transition(Ms.get(i),     Cs.get(i),     Bs.get(i),
                              Ms.get(i + 1), Cs.get(i + 1), Bs.get(i + 1)).toArray(BoolExpr[]::new));
    }
    if (solver.check() == Status.SATISFIABLE) {
        Model model = solver.getModel();
        Map<String, Integer> solution = new LinkedHashMap<>();
        for (int i = 0; i <= n; i++) {
            solution.put("M" + i, Integer.parseInt(model.eval(Ms.get(i), true).toString()));
            solution.put("C" + i, Integer.parseInt(model.eval(Cs.get(i), true).toString()));
            solution.put("B" + i, Integer.parseInt(model.eval(Bs.get(i), true).toString()));
        }
        return solution;
    }
    return null;
}

The function `findSolution` computes a solution to the problem of saving the infidels.

In [ ]:
Map<String, Integer> findSolution() {
    int n = 1;
    while (true) {
        System.out.println("Trying n=" + n + "...");
        var solution = missionariesCSP(n);
        if (solution != null) {
            return solution;
        }
        n += 2;
    }
}

It takes less than a second to solve the problem.

In [ ]:
long start   = System.currentTimeMillis();
var solution = findSolution();
System.out.println("Missionaries Solving: " + (System.currentTimeMillis() - start) + " ms");
solution

In [ ]:
void showSolution(Map<String, Integer> sol) {
    int n = sol.size() / 3 - 1;
    for (int i = 0; i <= n; i++) {
        int M = sol.get("M" + i);
        int C = sol.get("C" + i);
        int B = sol.get("B" + i);
        String west = "😇".repeat(M) + "🥷".repeat(C);
        String east = "😇".repeat(3 - M) + "🥷".repeat(3 - C);
        int riverWidth = 28;
        System.out.println(west + " ".repeat(riverWidth) + east);
        if (B == 1 && i < n) {
            int MB = sol.get("M" + i) - sol.get("M" + (i + 1));
            int CB = sol.get("C" + i) - sol.get("C" + (i + 1));
            System.out.println(" ".repeat(12) + ">>> " + "😇".repeat(MB) + "🥷".repeat(CB) + " >>>");
        } else if (i < n) {
            int MB = sol.get("M" + (i + 1)) - sol.get("M" + i);
            int CB = sol.get("C" + (i + 1)) - sol.get("C" + i);
            System.out.println(" ".repeat(12) + "<<< " + "😇".repeat(MB) + "🥷".repeat(CB) + " <<<");
        }
    }
}

In [ ]:
showSolution(solution);

The function `showAnimatedSolution` displays an animation of the solution.  The animation is implemented in *JavaScript*, which is executed by the browser.  The Java code only inserts the data describing the solution into the HTML code: The placeholders `@uid@`, `@n@`, and `@states@` are replaced by a unique identifier, the number of crossings, and the list of states, respectively.  The list of states is given in *JSON* notation.

In [ ]:
void showAnimatedSolution(Map<String, Integer> sol) {
    if (sol == null) {
        System.out.println("No solution provided.");
        return;
    }
    int n = sol.size() / 3 - 1;
    // Prepare the state data for the frontend JavaScript
    var states = new StringJoiner(", ", "[", "]");
    for (int i = 0; i <= n; i++) {
        states.add("{\"M\": %d, \"C\": %d, \"B\": %d}".formatted(sol.get("M" + i), sol.get("C" + i), sol.get("B" + i)));
    }
    // Generate a unique ID suffix to avoid element collisions if run multiple times
    String uid = Integer.toString(new Random().nextInt(Integer.MAX_VALUE), 36);
    String htmlContent = """
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-@uid@" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-@uid@" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-@uid@" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-@uid@" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of @n@</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-m-@uid@" x="75" y="100" font-size="28" text-anchor="middle">😇😇😇</text>
            <text id="west-c-@uid@" x="75" y="150" font-size="28" text-anchor="middle">🥷🥷🥷</text>
            
            <text id="east-m-@uid@" x="525" y="100" font-size="28" text-anchor="middle"></text>
            <text id="east-c-@uid@" x="525" y="150" font-size="28" text-anchor="middle"></text>

            <g id="boat-@uid@" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-@uid@" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = @states@;
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-@uid@');
        const westM = document.getElementById('west-m-@uid@');
        const westC = document.getElementById('west-c-@uid@');
        const eastM = document.getElementById('east-m-@uid@');
        const eastC = document.getElementById('east-c-@uid@');
        const boatP = document.getElementById('boat-p-@uid@');
        const stepLabel = document.getElementById('label-step-@uid@');
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            // DIESEL ENGINE SIMULATION:
            // A sawtooth wave at ~12 Hz creates a rapid "clicking/knocking" 
            // sound that mimics the slow chug of a diesel engine.
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            
            // Low-pass filter to muffle the harsh highs and keep it sounding bassy/heavy
            filter.type = 'lowpass';
            filter.frequency.value = 150;

            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            // Using a slightly higher gain (0.6) because extreme low frequencies are perceived as quieter
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4); // Slower fade out for realism
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            westM.textContent = '😇'.repeat(s.M);
            westC.textContent = '🥷'.repeat(s.C);
            eastM.textContent = '😇'.repeat(3 - s.M);
            eastC.textContent = '🥷'.repeat(3 - s.C);
            boatP.textContent = '';
            
            boat.style.transform = s.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of @n@';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const mPass = Math.abs(s0.M - s1.M);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.B === 1) { 
                westM.textContent = '😇'.repeat(s0.M - mPass);
                westC.textContent = '🥷'.repeat(s0.C - cPass);
            } else { 
                eastM.textContent = '😇'.repeat((3 - s0.M) - mPass);
                eastC.textContent = '🥷'.repeat((3 - s0.C) - cPass);
            }
            boatP.textContent = '😇'.repeat(mPass) + '🥷'.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-@uid@').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-@uid@').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-@uid@').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    
    """.replace("@uid@", uid)
       .replace("@n@", String.valueOf(n))
       .replace("@states@", states.toString());
    display(htmlContent, "text/html");
}

In [ ]:
showAnimatedSolution(solution);